# Data quality checks tutorial

This tutorial shows you how to create and run a DQ check on a set of instruments that were recently loaded into LUSID.

## Section 1: Create and test a check definition
In section 1, the notebook follows the steps explained in [How do I create and run a data quality check?](https://support.lusid.com/docs/how-do-i-create-and-run-a-data-quality-check).
1. Set up LUSID instruments that we plan to run DQ checks on. 
2. Create rulesets (the basis of DQ checks).
3. Create a check definition that defines which data to run the checks on and any limits for each rule.
4. Run the DQ check and handle the results.

## Section 2: Set up a DQ check workflow
In section 2, the notebook follows the steps explained in [How do I set up a data quality check workflow?](https://support.lusid.com/docs/how-do-i-set-up-a-data-quality-check-workflow).
1. Create exception task definition to handle DQ check results (breaches).
2. Create DQ check task definition, the main task we'll run whenever we want to perform a DQ check.
3. Kick off a task test run.
4. Inspect the results. 

# Setup
Build the LUSID and Workflow APIs and create some test data to run a DQ check on

In [ ]:
# import the latest SDK version
#!pip3 install -U lusid-sdk
#!pip3 install -U lusid-workflow-sdk

In [ ]:
# Set up LUSID API for creating test data and check definitions
import os
import pandas as pd
import logging
logging.basicConfig(level = logging.INFO)

import lusid as lu
import lusid.api as la
import lusid.models as lm

from lusidjam import RefreshingToken
import lusid.extensions as le
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.jupyter_tools import StopExecution
from finbourne_sdk_utils.lpt.lpt import to_date

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically

api_url = os.getenv("FBN_API_URL") # or "https://<your-domain>.lusid.com/api" 
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs

lusid_config_loaders=[
    le.ArgsConfigurationLoader(api_url=api_url ,access_token = RefreshingToken(), app_name = "LusidJupyterNotebook"),
    le.EnvironmentVariablesConfigurationLoader(),
    le.SecretsFileConfigurationLoader(secrets_path)]
lusid_api_factory = le.SyncApiClientFactory(config_loaders=lusid_config_loaders)

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

from pprint import pprint
checkDefinitions_api = lusid_api_factory.build(la.CheckDefinitionsApi)

list_check_definitions_response = checkDefinitions_api.list_check_definitions()

display(list_check_definitions_response)

In [ ]:
# Set up Workflow API for automating the DQ checks with tasks
import lusid_workflow as lw
import lusid_workflow.api as lwa
import lusid_workflow.models as lwm
import lusid_workflow.extensions as lwe
import time

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically

workflow_api_url = os.getenv("FBN_WORKFLOW_API_URL") # or "https://<your-domain>.lusid.com/workflow" 
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Build workflow API client using the same factory pattern
workflow_config_loaders = [
    lwe.ArgsConfigurationLoader(
        api_url=workflow_api_url,
        access_token=RefreshingToken(),
        app_name="LusidJupyterNotebook"
    ),
    lwe.EnvironmentVariablesConfigurationLoader(),
    lwe.SecretsFileConfigurationLoader(secrets_path)
]
workflow_api_factory = lwe.SyncApiClientFactory(config_loaders=workflow_config_loaders)
task_definitions_api = workflow_api_factory.build(lwa.TaskDefinitionsApi)
tasks_api = workflow_api_factory.build(lwa.TasksApi)

try:
    task_definitions_api.list_task_definitions()
    print("Successfully connected to the Workflow API.")
except lw.ApiException as e:
    print(f"Failed to connect to the Workflow API: {e}")

In [ ]:
# Obtain the Instruments API
instruments_api=lusid_api_factory.build(la.InstrumentsApi)

# Read assets into pandas dataframe from securities file and show it
assets_df = pd.read_csv("data/assets.csv", keep_default_na = False)
display(assets_df)

## Set up instruments in LUSID

In [ ]:
# Specify a unique scope and code to segregate data in this example
module_scope = "Finbourne-Examples"
module_code = "DQ-Check"
print(f"'{module_scope}\{module_code}' scope and code created.")

In [ ]:
# Create a dictionary of instrument definitions from asset data
definitions = {}

# Iterate over each row in the assets dataframe
for index, asset in assets_df.iterrows():
    
    # Map identifier columns to case-sensitive LUSID identifier names       
    identifiers = {
        # Unique identifiers
        "Figi": lm.InstrumentIdValue(value = asset["figi"]),
        "ClientInternal": lm.InstrumentIdValue(value = asset["internal_id"]),
        # Non-unique identifiers
        "Isin": lm.InstrumentIdValue(value = asset["isin"]),
        "Ticker": lm.InstrumentIdValue(value = asset["ticker"])  
    }
                                       
    # Model equities
    if asset["security_type"] == "equity":
        # Create definitions
        definitions[asset["instrument_name"]] = lm.InstrumentDefinition(
            name = asset["instrument_name"],
            identifiers = identifiers,
            definition = lm.Equity(
                instrument_type = "Equity",
                dom_ccy = asset["currency"],
                identifiers = {}
            )
        )
    # Model bonds
    elif asset["security_type"] == "govt bond":
        definitions[asset["instrument_name"]] = lm.InstrumentDefinition(
            name = asset["instrument_name"],
            identifiers = identifiers,
            definition = lm.Bond(
                instrument_type = "Bond",
                start_date = "2021-01-01T00:00:00Z",
                maturity_date = asset["maturity_date"],
                dom_ccy = asset["currency"],
                flow_conventions = lm.FlowConventions(
                    currency = asset["currency"],
                    payment_frequency = "6M",
                    day_count_convention = "ActualActual",
                    roll_convention = "NoAdjustment",
                    payment_calendars = [],
                    reset_calendars = [],
                    settle_days = 0,
                    reset_days = 0
                ),
                # Note that a bond instrument is unitised and the quantity specified in bond transactions
                principal = 1,
                # Specified as a number rather than percentage, eg. so a bond paying 0.375% is specified as 0.00375
                coupon_rate = float(asset["coupon"])
            )
        )
    
# Upsert instruments to a custom scope in LUSID
upsert_instruments_response = instruments_api.upsert_instruments(
    request_body = definitions, 
    scope = f"{module_scope}{module_code}",
)

# Upsert instruments to a custom scope in LUSID
upsert_instruments_response = instruments_api.upsert_instruments(
    request_body = definitions, 
    scope = f"{module_scope}{module_code}",
)

# Transform API response to a dataframe and show internally-generated unique LUID for each mastered instrument
upsert_instruments_response_df = lusid_response_to_data_frame(list(upsert_instruments_response.values.values()))
display(upsert_instruments_response_df[["name", "lusidInstrumentId"]])


## Create  property definitions and add to some instruments
Let's imagine we want to check that all instruments with the `Instrument/<scope>/RefreshData` property set to `True` have the `Instrument/<scope>/Issuer` property populated. We'll first need to create those property definitions:

In [ ]:
# Decorate instruments with properties
property_definitions_api = lusid_api_factory.build(lu.PropertyDefinitionsApi)

try:
    issuer_prop_definition = property_definitions_api.create_property_definition(
        create_property_definition_request=lu.CreatePropertyDefinitionRequest(
            domain="Instrument",
            scope=f"{module_scope}{module_code}",
            code="Issuer",
            display_name="Issuer",
            property_description="The issuer of the security",
            data_type_id=lm.ResourceId(
                scope="system",
                code="string"
            ),
            life_time="Perpetual",
            constraint_style="Property"
        )
    )
    print(f"Property 'Issuer' created successfully.")
except lu.ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print("Property 'Issuer' already exists, skipping creation.")
    else:
        raise

In [ ]:
try:
    refreshData_prop_definition = property_definitions_api.create_property_definition(
        create_property_definition_request=lu.CreatePropertyDefinitionRequest(
            domain="Instrument",
            scope=f"{module_scope}{module_code}",
            code="RefreshData",
            display_name="RefreshData",
            property_description="Tags data that should be included in integration runs",
            data_type_id=lm.ResourceId(
                scope="system",
                code="string"
            ),
            life_time="Perpetual",
            constraint_style="Property"
        )
    )
    pprint(refreshData_prop_definition)
except lu.ApiException as e:
    if e.status == 400 and "PropertyAlreadyExists" in str(e.body):
        print("Property 'RefreshData' already exists, skipping creation.")
    else:
        raise

We'll add the `Instrument/<scope>/Issuer` property to just one of our instruments. This means that all instruments other than `BBG000BDWPY0` should fail the DQ check when we set it up.

In [ ]:
result = instruments_api.upsert_instruments_properties(
    scope=f"{module_scope}{module_code}",
    upsert_instrument_property_request=[
        lu.UpsertInstrumentPropertyRequest(
            identifier_type="Figi",
            identifier="BBG000BDWPY0",
            properties=[
                lu.ModelProperty(
                    key=f"Instrument/{module_scope}{module_code}/Issuer",
                    value=lu.PropertyValue(
                        label_value="NEXT PLC"
                    )
                )
            ]
        )
    ]
)

print(f"Successfully upserted properties as at: {result.as_at_date}")

We'll add the `Instrument/<scope>/RefreshData` property to all of our instruments to be able to later indicate that we want to include all of them in runs of the DQ check.

In [ ]:
# Define the property to add
refreshData_true = lu.ModelProperty(
    key=f"Instrument/{module_scope}{module_code}/RefreshData",
    value=lu.PropertyValue(
        label_value="True"
    )
)

# Build a list of upsert requests for all instruments in the CSV
upsert_requests = []
for _, row in assets_df.iterrows():
    upsert_requests.append(
        lu.UpsertInstrumentPropertyRequest(
            identifier_type="Figi",
            identifier=row["figi"],  # Adjust column name if different
            properties=[refreshData_true]
        )
    )

# Upsert the property to all instruments in a single API call
add_refreshData_property = instruments_api.upsert_instruments_properties(
    scope=f"{module_scope}{module_code}",
    upsert_instrument_property_request=upsert_requests
)

print(f"Property added to {len(upsert_requests)} instruments")
print(add_refreshData_property)

# Section 1: Create and test a check definition
## Step 1: Create a check definition that contains empty rulesets

In [ ]:
rule_set_key = "instrument-properties-checks"

rule_sets = [lm.UpdateCheckDefinitionRuleSet(
    rule_set_key = rule_set_key,
    display_name = "Instrument properties checks ruleset",
    description = "A set of rules to apply to instruments assigned for data refreshes to check for appropriate properties.",
    rule_set_filter = f"Properties[Instrument/{module_scope}{module_code}/RefreshData] exists"
)]

try:
    create_check_definition_request = lm.CreateCheckDefinitionRequest(
    id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-instrument-properties"
        ),
    display_name = "Instruments check",
    description = "A check definition to validate instruments are populated with the correct properties",
    dataset_schema = lm.CheckDefinitionDatasetSchema(
        type = "LusidEntity",
        entity_type = "Instrument"    
    ),
    rule_sets = rule_sets
)
     
    create_check_definition_response = checkDefinitions_api.create_check_definition(create_check_definition_request=create_check_definition_request)

    pprint(create_check_definition_response)
except lu.ApiException as e:
    if e.status == 400 and "EntityWithIdAlreadyExists" in str(e.body):
        print(f"Check definition '{module_code}-instrument-properties' already exists in scope '{module_scope}', skipping creation.")
    else:
        raise

# Step 2: Upsert rules to the check definition

In [ ]:
# We want to create the following rule:
# "Check instruments have the property Instrument/<scope>/Issuer"

check_definition_rule = lm.CheckDefinitionRule(
    rule_key = "issuer-exists",
    display_name = "Issuer exists check",
    description = f"Checks whether an instrument is decorated with the Instrument/{module_scope}{module_code}/Issuer property",
    rule_formula = f"properties[Instrument/{module_scope}{module_code}/Issuer] exists",
    severity = 1   
)

try:
    upsert_data_quality_rule = [lm.UpsertDataQualityRule(
        rule_set_key=rule_set_key,
        rule=check_definition_rule
    )]
     
    upsert_rule_response = checkDefinitions_api.upsert_rules(scope=module_scope, code=f"{module_code}-instrument-properties", upsert_data_quality_rule=upsert_data_quality_rule)
    
    print(f"Successfully upserted rule '{check_definition_rule.rule_key}' to ruleset '{rule_set_key}'.")
    pprint(upsert_rule_response)
except lu.ApiException as e:
    print(f"Error creating ruleset: {e}")

## Step 2: Run the check on some data

Running the check, we can see that LUSID outputs a result for each instrument with the `RefreshData` property set to `True` that is missing the `Instrument/<scope>/Issuer` property.

In [ ]:

try:
    run_check_request = lm.RunCheckRequest(
        lusid_entity_dataset = lm.LusidEntityDataset(
            # scope means instrument scope to check within
            scope = f"{module_scope}{module_code}",
            selector_attribute = f"Properties[Instrument/{module_scope}{module_code}/RefreshData]",
            selector_value = "True",
            return_identifier_key = "Instrument/default/ClientInternal"
        ),
        limit_individual_breaches_per_rule = 100
)
     
    run_check_response = checkDefinitions_api.run_check_definition(scope=module_scope, code=f"{module_code}-instrument-properties", run_check_request=run_check_request)

    results = run_check_response.data_quality_check_results
    total   = len(results)
    print(f"Check complete: {total} results (breaches)")
except lu.ApiException as e:
    print(f"Error running check: {e}")

# Section 2: Set up DQ check workflow

In [ ]:
# Helper functions for compact list construction
def fields(*items):
    return [lwm.TaskFieldDefinition(name=n, type=t, display_name=d) for n, t, d in items]

def states(*names):
    return [lwm.TaskStateDefinition(name=n, display_name=n, description=n) for n in names]

def triggers(*names):
    return [lwm.TransitionTriggerDefinition(name=n, trigger=lwm.TriggerSchema(type="External")) for n in names]

def transitions(*items):
    return [lwm.TaskTransitionDefinition(from_state=f, to_state=t, trigger=tr, **kw) for f, t, tr, *rest in items for kw in [rest[0] if rest else {}]]

def map_from(field):
    return lwm.FieldMapping(map_from=field, set_to=None)

## Step 1: Create exception task definition

We need to create this task definition first so we can reference it in the main DQ check task definition. 

LUSID will create one exception task for each DQ check result (breach).

In [ ]:
trigger_parent_task_action_instance = lwm.TriggerParentTaskAction(type="TriggerParentTask", trigger="resolve")


exception_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-exception"),
    display_name="DQ Exception",
    description="An exception returned by a data quality check.",
    states=states("Pending", "InProgress", "Resolved"),
    field_schema=fields(
        ("checkDefinitionScope",       "String",   "CheckDef Scope"),
        ("checkDefinitionCode",        "String",   "CheckDef Code"),
        ("checkDefinitionDisplayName", "String",   "CheckDef Name"),
        ("checkRunAsAt",               "DateTime", "Run AsAt"),
        ("resultType",                 "String",   "Result Type"),
        ("rulesetKey",                 "String",   "Ruleset Key"),
        ("rulesetDisplayName",         "String",   "Ruleset Name"),
        ("ruleKey",                    "String",   "Rule Key"),
        ("ruleDisplayName",            "String",   "Rule Name"),
        ("ruleDescription",            "String",   "Rule Description"),
        ("ruleFormula",                "String",   "Rule Formula"),
        ("severity",                   "Decimal",  "Severity"),
        ("resultId",                   "String",   "Result Tracking ID"),
        ("entityType",                 "String",   "Entity Type"),
        ("instrumentAsAt",             "DateTime", "Instrument As At"),
        ("instrumentEffectiveAt",      "DateTime", "Instrument Effective At"),
        ("instrumentScope",            "String",   "Instrument Scope"),
        ("identifierType",             "String",   "Identifier Type"),
        ("identifierValue",            "String",   "Identifier Value"),
        ("instrumentName",             "String",   "Instrument Name"),
        ("entityUniqueId",             "String",   "Entity Unique ID"),
        ("countRuleBreaches",          "Decimal",  "Number of Breaches"),
        ("errorDetail",                "String",   "Error Message"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "resolve"),
    actions=[
        lwm.ActionDefinition(
            name="resolve-parent",
            action_details=lwm.ActionDetails(trigger_parent_task_action_instance)
        )
    ],
    transitions=transitions(
        ("Pending",    "InProgress", "start"),
        ("InProgress", "Resolved",   "resolve", {"action": "resolve-parent"}),
    )
)

try:
    exception_response = task_definitions_api.create_task_definition(
        create_task_definition_request=exception_task_def
    )
    print(f"Task definition created successfully. Scope: {exception_response.id.scope}, Code: {exception_response.id.code}")
except lw.ApiException as e:
    print(f"Error creating exception task definition: {e}")

## Step 2: Create DQ check task definition

This is the parent task definition that we can run each time we want to perform a DQ check on some data.

We pass in the scope and code of our check definition to LUSID's built-in DQ check worker `worker_id=lwm.ResourceId(scope="default", code="LusidEntityDataQuality")`. By default, the worker can kick off runs of the specified DQ check and turn each result into its own exception task for resolution.


In [ ]:
child_task_fields = {k: map_from(v) for k, v in [
    ("checkDefinitionScope",       "checkDefinitionScope"),
    ("checkDefinitionCode",        "checkDefinitionCode"),
    ("checkDefinitionDisplayName", "checkDefinitionDisplayName"),
    ("checkRunAsAt",               "checkRunAsAt"),
    ("resultType",                 "resultType"),
    ("rulesetKey",                 "rulesetKey"),
    ("rulesetDisplayName",         "rulesetDisplayName"),
    ("ruleKey",                    "ruleKey"),
    ("ruleDisplayName",            "ruleDisplayName"),
    ("ruleDescription",            "ruleDescription"),
    ("ruleFormula",                "ruleFormula"),
    ("resultId",                   "resultId"),
    ("entityType",                 "lusidEntityType"),
    ("instrumentAsAt",             "lusidEntityAsAt"),
    ("instrumentEffectiveAt",      "lusidEntityEffectiveAt"),
    ("instrumentScope",            "lusidEntityScope"),
    ("identifierType",             "lusidEntityIdentifierKey"),
    ("identifierValue",            "lusidEntityIdentifierValue"),
    ("instrumentName",             "lusidEntityDisplayName"),
    ("entityUniqueId",             "lusidEntityUniqueId"),
    ("severity",                   "severity"),
    ("countRuleBreaches",          "countRuleBreaches"),
    ("errorDetail",                "errorDetail"),
]}

dq_check_task_def = lwm.CreateTaskDefinitionRequest(
    id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckInstruments"),
    display_name="DQ Check Instruments",
    description="Runs data quality checks for instruments.",
    states=states("Pending", "InProgress", "ExceptionManagement", "Complete", "Error"),
    field_schema=fields(
        ("checkDefinitionScope", "String",  "CD Scope"),
        ("checkDefinitionCode",  "String",  "CD Code"),
        ("instrumentScope",      "String",  "Instrument Scope"),
        ("selectorAttribute",    "String",  "Selector Attribute"),
        ("selectorValue",        "String",  "Selector Value"),
        ("preferredIdentifier",  "String",  "Preferred Identifier"),
        ("ruleBreachLimit",      "Decimal", "Rule Breach Limit"),
    ),
    initial_state=lwm.InitialState(name="Pending", required_fields=[]),
    triggers=triggers("start", "exceptions", "no_exceptions", "resolve", "error"),
    actions=[
        lwm.ActionDefinition(
            name="run-checks",
            action_details=lwm.ActionDetails(
                lwm.RunWorkerAction(
                    type="RunWorker",
                    worker_id=lwm.ResourceId(scope="default", code="LusidEntityDataQuality"),
                    worker_parameters={
                        "checkDefinitionScope":           map_from("checkDefinitionScope"),
                        "checkDefinitionCode":            map_from("checkDefinitionCode"),
                        "dataSetScope":                   map_from("instrumentScope"),
                        "selectorAttribute":              map_from("selectorAttribute"),
                        "selectorValue":                  map_from("selectorValue"),
                        "returnIdentifierKey":            map_from("preferredIdentifier"),
                        "limitIndividualBreachesPerRule": map_from("ruleBreachLimit"),
                    },
                    worker_status_triggers=lwm.WorkerStatusTriggers(
                        started=None,
                        completed_with_results="exceptions",
                        completed_no_results="no_exceptions",
                        failed_to_start="error",
                        failed_to_complete="error"
                    ),
                    child_task_configurations=[
                        lwm.ResultantChildTaskConfiguration(
                            task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-exception"),
                            map_stacking_key_from="resultId",
                            child_task_fields=child_task_fields,
                            result_matching_pattern=None,
                            initial_trigger=None
                        )
                    ]
                )
            )
        )
    ],
    transitions=transitions(
        ("Pending",             "InProgress",          "start",         {"action": "run-checks"}),
        ("InProgress",          "Complete",            "no_exceptions"),
        ("InProgress",          "ExceptionManagement", "exceptions"),
        ("ExceptionManagement", "Complete",            "resolve",       {"guard": "childTasks all (state eq 'Resolved')"}),
        ("InProgress",          "Error",               "error"),
    )
)

try:
    dq_check_response = task_definitions_api.create_task_definition(
        create_task_definition_request=dq_check_task_def
    )
    print(f"Task definition created successfully. Scope: {dq_check_response.id.scope}, Code: {dq_check_response.id.code}")
except lw.ApiException as e:
    print(f"Error creating DQ check task definition: {e}")

## Step 3: Kick off a run of the task to run the DQ check on some data

In [ ]:
create_task_request = lwm.CreateTaskRequest(
    task_definition_id=lwm.ResourceId(scope=module_scope, code=f"{module_code}-CheckInstruments"),
    correlation_ids=[],
    fields=[
        lwm.TaskInstanceField(
            name="checkDefinitionScope",
            value=module_scope
        ),
        lwm.TaskInstanceField(
            name="checkDefinitionCode",
            value=f"{module_code}-instrument-properties"
        ),
        lwm.TaskInstanceField(
            name="instrumentScope",
            value=f"{module_scope}{module_code}"
        ),
        lwm.TaskInstanceField(
            name="selectorAttribute",
            value=f"Properties[Instrument/{module_scope}{module_code}/RefreshData]"
        ),
        lwm.TaskInstanceField(
            name="selectorValue",
            value="True"
        ),
        lwm.TaskInstanceField(
            name="preferredIdentifier",
            value="Instrument/default/ClientInternal"
        ),
        lwm.TaskInstanceField(
            name="ruleBreachLimit",
            value="100"
        ),
    ]
)

try:
    create_task_response = tasks_api.create_task(
        create_task_request=create_task_request,
        trigger="start"
    )
    task_id = create_task_response.id
    print(f"Task created successfully. ID: {task_id}")
except lw.ApiException as e:
    print(f"Error creating task: {e}")

If necessary, allow time for LUSID to gather all results and create an exception child task for each breach:

In [ ]:
print("Waiting 30 seconds for LUSID to run checks and find breaches...")
time.sleep(30)
print("Done waiting, retrieving child tasks...")

try:
    all_child_tasks = []
    response = tasks_api.list_tasks(
        filter=f"ultimateParentTask.id eq '{task_id}'"
    )
    all_child_tasks.extend([t for t in response.values if t.parent_task is not None])

    while response.next_page:
        response = tasks_api.list_tasks(
            filter=f"ultimateParentTask.id eq '{task_id}'",
            page=response.next_page
        )
        all_child_tasks.extend([t for t in response.values if t.parent_task is not None])

    print(f"Found {len(all_child_tasks)} child tasks.")
except lw.ApiException as e:
    print(f"Error retrieving child tasks: {e}")

We can see each of the instruments that failed the DQ check. You can then resolve breaks and manage your tasks via the LUSID web app. [Read more.](https://support.lusid.com/docs/how-do-i-set-up-a-data-quality-check-workflow)

In [ ]:
def display_exception_tasks(tasks):
    rows = []
    for task in tasks:
        field_dict = {f.name: f.value for f in task.fields}
        rows.append({
            "Task ID":         task.id,
            "State":           task.state,
            "Instrument Name": field_dict.get("instrumentName"),
            "Identifier":      f"{field_dict.get('identifierType')}: {field_dict.get('identifierValue')}",
            "Rule":            field_dict.get("ruleDisplayName"),
            "Rule Formula":    field_dict.get("ruleFormula"),
            "Result Type":     field_dict.get("resultType"),
            "Severity":        field_dict.get("severity"),
            "Ruleset":         field_dict.get("rulesetDisplayName"),
            "Check Run At":    field_dict.get("checkRunAsAt"),
        })
    display(pd.DataFrame(rows))

try:
    all_exception_tasks = []
    response = tasks_api.list_tasks(
        filter=f"ultimateParentTask.id eq '{task_id}'"
    )
    all_exception_tasks.extend([t for t in response.values if t.parent_task is not None])

    while response.next_page:
        response = tasks_api.list_tasks(
            filter=f"ultimateParentTask.id eq '{task_id}'",
            page=response.next_page
        )
        all_exception_tasks.extend([t for t in response.values if t.parent_task is not None])

    print(f"Found {len(all_exception_tasks)} exception tasks.")
    display_exception_tasks(all_exception_tasks)
except lw.ApiException as e:
    print(f"Error retrieving child tasks: {e}")